# Importing Libraries

In [ ]:
from dataclasses import dataclass
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from peft import get_peft_model, SftConfig, SftTrainer, TaskType
from utils import *

# Configuration

In [ ]:
@dataclass()
class Config:
    model_id: str = "meta-llama/Llama-3.2-1B-Instruct"
    peft_density: float = 0.01
    peft_selection_algorithm: str = "rigl" # or "sm3" for moment approximation SFT
    peft_target_modules = ["q_proj", "o_proj", "v_proj", "k_proj", "gate_proj", "up_proj", "down_proj"]


    dataset_id: str = "SirNeural/flan_v2"


    dtype: str = "bfloat16"

    seed: int = 42

config = Config()

In [ ]:
set_seed(config.seed)

# Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    device_map="auto",
    dtype="auto"
)
tokenizer = AutoTokenizer.from_pretrained(
    config.model_id,
    use_fast=True
)

In [ ]:
print(model)

# Dataset

In [ ]:
dataset = load_dataset(config.dataset_id)

# Training

In [ ]:
peft_config = SftConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    density=config.peft_density,
    selection_algorithm=config.peft_selection_algorithm,
    target_modules=config.peft_target_modules
)
model = get_peft_model(model, peft_config)

In [ ]:
model.print_trainable_parameters()

In [ ]:
trainer_cls = SftTrainer(MyTrainer) # MyTrainer = Trainer or any subclass thereof
trainer = trainer_cls(
    model=model,
    args=training_args,
    sft_config=peft_config,
)

In [ ]:
trainer.train()